# Chapter 2 · 제조 수요 SARIMA

**연구질문:** 일별 제조 수요에서 SARIMA는 계절 나이브보다 낫고, 잔차에 자기상관이 남는가?

> 위에서 아래로 실행하면 끝까지 동작합니다. 온라인 실패 시 합성 대체(`DATA_KIND` 표시).

## 1. 핵심 개념·수식

**ARIMA$(p,d,q)$:** $\phi(L)(1-L)^d y_t = c + \theta(L)\varepsilon_t$, $\ \varepsilon_t \sim WN(0,\sigma^2)$.

**SARIMA$(p,d,q)(P,D,Q)_m$:** 계절 차분 $(1-L^m)^D$와 계절 AR/MA 다항식 $\Phi(L^m),\Theta(L^m)$을 추가한다.

**차수 선택:** $\mathrm{AIC}=2k-2\ln\hat{L}$. AIC는 **적합 내 기준**이며 외표본 성능을 보장하지 않는다. 항상 시간순 홀드아웃에서 기준선과 비교한다.

**예측 오차 지표:** $\mathrm{MAE}$, $\mathrm{RMSE}$, $\mathrm{MASE}=\dfrac{\mathrm{MAE}}{\frac{1}{T-1}\sum_{t=2}^{T}|y_t-y_{t-1}|}$ (분모는 학습구간 1단계 나이브의 평균절대오차; 1보다 작으면 학습구간 나이브보다 낫다).

**잔차 진단:** 잔차가 백색잡음이면 Ljung–Box $Q$ 검정에서 기각되지 않아야 한다.

In [ ]:
import os, io, json, subprocess, sys, warnings, urllib.request, urllib.parse
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
# 네트워크 없이 합성자료로만 실행하려면 환경변수 TSLAB_OFFLINE=1 로 실행하거나 아래를 False로 바꾼다.
USE_LIVE_DATA = os.environ.get("TSLAB_OFFLINE") != "1"
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True,
                     "axes.spines.top": False, "axes.spines.right": False})

def ensure(pkg, mod=None):
    # 없으면 설치를 시도한다. 실패해도 노트북은 계속 실행되고 해당 선택 기능만 건너뛴다.
    try:
        __import__(mod or pkg); return True
    except Exception:
        if not USE_LIVE_DATA:
            return False
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True, timeout=300)
            __import__(mod or pkg); return True
        except Exception as e:
            print(f"[건너뜀] {pkg} 설치/가져오기 실패: {type(e).__name__}")
            return False

def fetch_bytes(url, timeout=60):
    req = urllib.request.Request(url, headers={"User-Agent": "ts-lab-course/1.0 (education)"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

# ---- 평가 지표와 기준선 ----
def mae(a, f):  return float(np.mean(np.abs(np.asarray(a, float) - np.asarray(f, float))))
def rmse(a, f): return float(np.sqrt(np.mean((np.asarray(a, float) - np.asarray(f, float)) ** 2)))
def mase(a, f, train, m=1):
    train = np.asarray(train, float)
    scale = np.mean(np.abs(train[m:] - train[:-m]))
    return mae(a, f) / scale
def naive_fc(train, h):  return np.repeat(np.asarray(train, float)[-1], h)
def snaive_fc(train, h, m):
    train = np.asarray(train, float)
    return np.tile(train[-m:], int(np.ceil(h / m)))[:h]
def drift_fc(train, h):
    train = np.asarray(train, float)
    slope = (train[-1] - train[0]) / (len(train) - 1)
    return train[-1] + slope * np.arange(1, h + 1)
def show(df, digits=4):
    print(df.round(digits).to_string())

In [ ]:
def synthetic_sensor(n_days=180, seed=SEED, n_faults=14):
    # NOT real data: 일주기 + 느린 드리프트 + 잡음, 고장 직전 12시간 동안 진동이 증가하는 전조 구간 포함
    rng = np.random.default_rng(seed); n = n_days * 24; t = np.arange(n)
    vib = 1.0 + 0.15 * np.sin(2 * np.pi * (t % 24) / 24) + 0.0005 * t + rng.normal(0, 0.05, n)
    temp = 60 + 3 * np.sin(2 * np.pi * (t % 24) / 24 - 0.5) + rng.normal(0, 0.6, n)
    fault = np.zeros(n, int)
    starts = np.sort(rng.choice(np.arange(72, n - 24), n_faults, replace=False))
    for s0 in starts:
        for k in range(12):
            vib[s0 - 12 + k] += 0.03 * (k + 1); temp[s0 - 12 + k] += 0.15 * (k + 1)
        vib[s0:s0 + 2] += 0.8; fault[s0] = 1
    idx = pd.date_range("2024-01-01", periods=n, freq="h")
    return pd.DataFrame({"vibration": vib, "temp": temp, "fault": fault}, index=idx)

def synthetic_demand(n=1095, seed=SEED + 2):
    rng = np.random.default_rng(seed); t = np.arange(n)
    mu = 500 + 0.15 * t + 40 * np.sin(2 * np.pi * t / 365.25) + 60 * (t % 7 < 5) + rng.normal(0, 12, n)
    return pd.Series(mu, index=pd.date_range("2021-01-01", periods=n, freq="D"), name="demand")

def load_manufacturing(which="sensor"):
    # 달력 시간이 있는 무료 제조 시계열은 드물다(카탈로그 3절). 기본은 합성 자료이며 NOT real data로 표시한다.
    if which == "demand":
        return dict(y=synthetic_demand(), m=7, freq="D", label="일별 제조 수요(합성)", unit="개",
                    kind="SYNTHETIC (NOT real data)")
    df = synthetic_sensor()
    return dict(y=df["vibration"], df=df, m=24, freq="h", label="진동 센서(합성)", unit="mm/s",
                kind="SYNTHETIC (NOT real data)")

D = load_manufacturing("demand")
y, DATA_KIND, M, FREQ = D["y"], D["kind"], D["m"], D["freq"]
print("DATA_KIND:", DATA_KIND); print("관측 수:", len(y), "| 기간:", y.index.min(), "~", y.index.max())

## 2. 분할과 기준선

In [ ]:
H = 28                                  # 예측 구간 길이(관측 단위)
LOG = False                              # 건수·양수 자료의 분산 안정화를 위한 log1p 사용 여부
s_all = y.astype(float)
z_all = np.log1p(s_all) if LOG else s_all   # 모형이 보는 척도
train_z, test_z = z_all.iloc[:-H].to_numpy(), z_all.iloc[-H:].to_numpy()
train, test = s_all.iloc[:-H].to_numpy(), s_all.iloc[-H:].to_numpy()
back = (lambda v: np.expm1(v)) if LOG else (lambda v: v)    # 모형 척도 -> 원척도
print(f"학습 {len(train)} | 검증 {len(test)} | log1p 사용: {LOG}")

In [ ]:
results = {}
def record(name, fc):
    fc = np.asarray(fc, float)
    results[name] = {"MAE": mae(test, fc), "RMSE": rmse(test, fc), "MASE": mase(test, fc, train, m=1)}
record("naive", naive_fc(train, H))
record("drift", drift_fc(train, H))
if M > 1 and M <= len(train) // 3:
    record(f"seasonal naive(m={M})", snaive_fc(train, H, M))
show(pd.DataFrame(results).T)

## 3. ARIMA / SARIMA

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import adfuller

d = 0 if adfuller(train_z, autolag="AIC")[1] < 0.05 else 1       # 단순 규칙(검정은 보조 수단)
cands = []
for p in range(3):
    for q in range(3):
        try:
            fit = ARIMA(train_z, order=(p, d, q)).fit()
            cands.append((fit.aic, p, q, fit))
        except Exception:
            continue
cands.sort(key=lambda t: t[0])
aic, p_, q_, best = cands[0]
print(f"선택 ARIMA({p_},{d},{q_})  AIC={aic:.1f}  (후보 {len(cands)}개 중 AIC 최소)")
fc = best.get_forecast(H)
mean_z, ci_z = np.asarray(fc.predicted_mean), np.asarray(fc.conf_int(alpha=0.05))
record(f"ARIMA({p_},{d},{q_})", back(mean_z))
lb = acorr_ljungbox(best.resid, lags=[10], return_df=True)["lb_pvalue"].iloc[0]
print("잔차 Ljung-Box(10) p =", round(float(lb), 4), "-> 작으면 잔차에 자기상관이 남음")

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
USE_SARIMA = M in (7, 12) and len(train_z) > 4 * M
if USE_SARIMA:
    try:
        sfit = SARIMAX(train_z, order=(1, d, 1), seasonal_order=(1, 1, 1, M),
                       enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
        sfc = sfit.get_forecast(H)
        record(f"SARIMA(1,{d},1)(1,1,1)_{M}", back(np.asarray(sfc.predicted_mean)))
        s_mean_z, s_ci_z = np.asarray(sfc.predicted_mean), np.asarray(sfc.conf_int(alpha=0.05))
        print("SARIMA 잔차 Ljung-Box p =", round(float(acorr_ljungbox(sfit.resid, lags=[10], return_df=True)["lb_pvalue"].iloc[0]), 4))
    except Exception as e:
        USE_SARIMA = False; print("SARIMA 적합 실패:", type(e).__name__, str(e)[:100])
else:
    print(f"M={M}: 이 노트북에서는 SARIMA를 건너뜁니다(주기가 7·12가 아니거나 표본이 짧음).")

## 4. 비교와 진단

In [ ]:
tab = pd.DataFrame(results).T.sort_values("MASE")
show(tab)
best_name = tab.index[0]
print("검증구간 MASE 최소:", best_name, "| 주의: 단일 검증구간이므로 우열은 불확실(→ Ch.7 walk-forward)")
fig, ax = plt.subplots(figsize=(11, 4))
idx_te = s_all.index[-H:]
ax.plot(s_all.index[-4 * H:], s_all.iloc[-4 * H:], label="actual")
ax.plot(idx_te, back(mean_z), label=f"ARIMA({p_},{d},{q_})")
ax.fill_between(idx_te, back(ci_z[:, 0]), back(ci_z[:, 1]), alpha=0.15)
if "USE_SARIMA" in globals() and USE_SARIMA:
    ax.plot(idx_te, back(s_mean_z), label="SARIMA")
ax.plot(idx_te, snaive_fc(train, H, M) if (M > 1 and M <= len(train) // 3) else naive_fc(train, H), "--", label="baseline")
ax.legend(); ax.set_title(f"{D['label']} 예측 | {DATA_KIND}"); plt.tight_layout(); plt.show()

## 3. 결과 해석 작성
1. 어떤 기준선이 가장 강했는지, 복잡한 모형이 그 기준선을 외표본에서 이겼는지(MASE<기준선) 적는다.
2. 잔차 Ljung–Box 결과가 모형 충분성에 대해 말하는 바를 적는다.
3. AIC 최소 모형과 외표본 최소 오차 모형이 다르면 그 이유(과적합, 구조변화)를 논의한다.
4. 한계: 단일 검증구간, 로그변환 역변환 편의, 구조변화. 합성자료 결과는 방법 시연이다.

**제출:** 기준선 포함 지표표, 예측 그래프, 잔차 진단, 해석 단락, 데이터 출처·한계.